Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/training/train-within-notebook/train-within-notebook.png)

# Train, register, and deploy a model with Scikit-learn

In this tutorial, we demonstrate how to use the Azure ML Python SDK to control all aspects of training and deploying a model with the `SKLearn` estimator directly from a notebook. We will train a regression model on the common [diabetes dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html) in order to predict disease progression based on physiological variables.

## Prerequisites

Go through the Configuration notebook to install the Azure Machine Learning Python SDK and create an Azure ML Workspace

In [ ]:
# Check core SDK version number
import azureml.core
print("SDK Version: ", azureml.core.VERSION)

## Diagnostics

Opt-in diagnostics for better experience, quality, and security of future releases.

In [ ]:
from azureml.telemetry import set_diagnostics_collection

set_diagnostics_collection(send_diagnostics=True)

## Initialize workspace

Initialize a [Workspace](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture#workspace) object from the existing workspace you created in the Prerequisites step. `Workspace.from_config()` creates a workspace object from the details stored in `config.json`.

In [ ]:
from azureml.core.workspace import Workspace

ws = Workspace.from_config()
print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

## Create AmlCompute

You will need to create a [compute target](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture#compute-target) for training your model. In this tutorial, we use Azure ML managed compute ([AmlCompute](https://docs.microsoft.com/azure/machine-learning/service/how-to-set-up-training-targets#amlcompute)) for our remote training compute resource.

As with other Azure services, there are limits on certain resources (e.g. AmlCompute) associated with the Azure Machine Learning service. Please read [this article](https://docs.microsoft.com/azure/machine-learning/service/how-to-manage-quotas) on the default limits and how to request more quota.

If we could not find the cluster with the given name, then we will create a new cluster here. We will create an `AmlCompute` cluster of `STANDARD_D2_V2` CPU VMs. This process is broken down into 3 steps:

1. Create the configuration (this step is local and only takes a second)
2. Create the cluster (this step will take about **20 seconds**)
3. Provision the VMs to bring the cluster to the initial size (of 1 in this case).

This step will take about **3-5 minutes** and is providing only sparse output in the process. Please make sure to wait until the call returns before moving to the next cell.

In [ ]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

# choose a name for your cluster
cluster_name = "cpu-cluster"

try:
    compute_target = ComputeTarget(workspace=ws, name=cluster_name)
    print('Found existing compute target')
except ComputeTargetException:
    print('Creating a new compute target...')
    compute_config = AmlCompute.provisioning_configuration(vm_size='STANDARD_D2_V2', 
                                                           max_nodes=4)

    # create the cluster
    compute_target = ComputeTarget.create(ws, cluster_name, compute_config)

    # can poll for a minimum number of nodes and for a specific timeout. 
    # if no min node count is provided it uses the scale settings for the cluster
    compute_target.wait_for_completion(show_output=True, min_node_count=None, timeout_in_minutes=20)

# use get_status() to get a detailed status for the current cluster. 
print(compute_target.get_status().serialize())

The above code retrieves a CPU compute target. Scikit-learn does not support GPU computing.

## Train model on the remote compute

Now that you have your data and training script prepared, you are ready to train on your remote compute. You can take advantage of Azure compute to leverage a CPU cluster.

### Create a project directory

Create a directory that will contain all the necessary code from your local machine that you will need access to on the remote resource. This includes the training script and any additional files your training script depends on.

### Prepare training script

Now you will need to create your training script. In this tutorial, the training script is already provided for you at `train_diabetes.py`. In practice, you should be able to take any custom training script as is and run it with Azure ML without having to modify your code.

Once your script is ready, copy the training script into your project directory.

### Create an experiment

Create an [Experiment](https://docs.microsoft.com/azure/machine-learning/service/concept-azure-machine-learning-architecture#experiment) to track all the runs in your workspace for this tutorial.

In [ ]:
from azureml.core import Experiment

experiment_name = 'train_diabetes'
experiment = Experiment(ws, name=experiment_name)

### Create a Scikit-learn estimator

The Azure ML SDK's `SKLearn` estimator enables you to easily submit Scikit-learn training jobs for single-node runs. The following code will define a single-node Scikit-learn job.

In [ ]:
from azureml.train.sklearn import SKLearn

script_params = {
    '--kernel': 'linear',
    '--penalty': 1.0,
}

estimator = SKLearn(source_directory=project_folder, 
                    script_params=script_params,
                    compute_target=compute_target,
                    entry_script='train_diabetes.py'
                   )

The `script_params` parameter is a dictionary containing the command-line arguments to your training script `entry_script`.

### Submit job

Run your experiment by submitting your estimator object. Note that this call is asynchronous.

In [ ]:
run = experiment.submit(estimator)

## Monitor your run

You can monitor the progress of the run with a Jupyter widget. Like the run submission, the widget is asynchronous and provides live updates every 10-15 seconds until the job completes.

In [ ]:
from azureml.widgets import RunDetails

RunDetails(run).show()

## Find and register model

Now, let's list the model files uploaded during the run.

In [ ]:
print(run.get_file_names())

We can then register the folder (and all files in it) as a model named `sklearn-diabetes-regr` under the workspace for deployment

In [ ]:
model = run.register_model(model_name='sklearn-diabetes-regr', model_path='outputs/model.joblib')

## Deploy the model in ACI

Now, we are ready to deploy the model as a web service running in Azure Container Instance, ACI. Azure Machine Learning accomplishes this by constructing a Docker image with the scoring logic and model baked in.

### Create scoring script

First, we will create a scoring script that will be invoked by the web service call.

+ Now that the scoring script must have two required functions, `init()` and `run(input_data)`.
    + In init(), you typically load the model into a global object. This function is executed only once when the Docker contianer is started.
    + In run(input_data), the model is used to predict a value based on the input data. The input and output to run uses joblib as the serialization and de-serialization format because it is the preferred format for Scikit-learn, but you are not limited to it.
    + Refer to the scoring script `sklearn_score.py` for this tutorial. Our web service will use this file to predict. When writing your own scoring script, don't forget to test it locally first before you go and deploy the web service.

### Deploy to ACI

First we create the environment and required configuration.
* Environment object which has list of packages needed for the model 
* Inference configuration which describes the scripts and environment to be used for the web service
* Deployment configuration which describes the resource requirements for the web service. In this example, we will be deploying to Azure Container Instances (ACI)

In [ ]:
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.environment import Environment
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice

env = Environment('deploytocloudenv')
env.python.conda_dependencies = CondaDependencies.create(conda_packages=['scikit-learn','joblib'], pip_packages=['azureml-defaults'])
inference_config = InferenceConfig(entry_script="sklearn_score.py", environment=env)

aciconfig = AciWebservice.deploy_configuration(cpu_cores=1,
                                               auth_enabled=True, # this flag generates API keys to secure access
                                               memory_gb=1,
                                               tags={'name': 'sklearn-diabetes', 'framework': 'Scikit-learn'},
                                               description='Ridge Regression with Scikit-learn')

Now we can deploy to ACI. The cell below will run for about **7-8 minutes** as we package up the required dependencies into a docker image and run that as a webservice. An HTTP endpoint accepting REST client calls will be exposed. You can call this endpoint to run your scoring script and do the predictions.

In [ ]:
%%time
from azureml.core.model import Model
from azureml.core.webservice import Webservice

service = Model.deploy(workspace=ws,
                       name='sklearn-diabetes',
                       models=[model],
                       inference_config=inference_config,
                       deployment_config=aciconfig)

service.wait_for_deployment(show_output=True)

In [ ]:
print(service.get_logs())

**Tip: If something goes wrong with the deployment, the first thing to look at is the logs from the service by running the following command:** `print(service.get_logs())`

This is the scoring web service endpoint: `print(service.scoring_uri)`

### Test the deployed model

Let's test the deployed model. Pick a random sample from the test set, and send it to the web service hosted in ACI for a prediction. 

In [ ]:
import json

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

import numpy as np

diabetes = load_diabetes()

# Use only one feature
diabetes_X = diabetes.data[:, np.newaxis, 2]

# Split the data into training/testing sets
diabetes_X_test = diabetes_X[-20:]
diabetes_y_test = diabetes.target[-20:]


# score the first row from the test set
test_samples = json.dumps({"data": (diabetes_X_test).tolist()})

result = service.run(input_data=test_samples)
print(result)

We can also construct a raw HTTP request to send to the service. Don't forget to add key to the HTTP header.

In [ ]:
import requests

# retreive the API keys. two keys were generated.
key1, Key2 = service.get_keys()

# create the required header
headers = {'Content-Type':'application/json', 'Authorization': 'Bearer ' + key1}

# post the request to the service and display the result
resp = requests.post(service.scoring_uri, test_samples, headers = headers)
print(resp.text)

### Residual Graph

One way to understand the behavior of your model is to see how the data performs against data with known results. This cell uses matplotlib to create a histogram of the residual values, or errors, created from scoring the test samples.

A good model should have residual values that cluster around 0 - that is, no error. Observing the resulting histogram can also show you if the model is skewed in any particular direction.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


# calculate residual
residual = result - diabetes_y_test

f, (a0, a1) = plt.subplots(1, 2, gridspec_kw={'width_ratios':[3, 1], 'wspace':0, 'hspace': 0})
f.suptitle('Residual Values', fontsize = 18)

f.set_figheight(6)
f.set_figwidth(14)

a0.plot(residual, 'bo', alpha=0.4)
a0.plot([0,90], [0,0], 'r', lw=2)
a0.set_ylabel('residue values', fontsize=14)
a0.set_xlabel('test data set', fontsize=14)

a1.hist(residual, orientation='horizontal', color='blue', bins=10, histtype='step')
a1.hist(residual, orientation='horizontal', color='blue', alpha=0.2, bins=10)
a1.set_yticklabels([])

plt.show()

### Delete ACI to clean up
Delete the ACI instance to stop the compute and any associated billing.

In [ ]:
service.delete()